## 3. Honest Model Quality Summary

### End of notebook phase

This notebook did two things:
- Rebuilt the selected model for each ticker on the training split.
- Chose a validation threshold that maximises F1, then saved the final model and updated hyperparameter file.

The important interpretation is that not every "optimal" threshold is actually useful. Some thresholds are mathematically best on the validation slice but collapse into overly aggressive positive predictions. For those cases, the default 0.50 threshold is still the safer production choice.

| Ticker | Model | ROC-AUC | Usable? | Notes |
|---|---|---:|---|---|
| ORWE_CA | LightGBM | 0.5895 | ✓ Best | Calibrated, balanced, genuine signal |
| HRHO_CA | XGBoost | 0.5502 | ✓ Good | Fixed and working, threshold near 0.50 |
| COMI_CA | XGBoost | 0.5234 | ⚠ Weak | Usable signal, default threshold is safer |
| SWDY_CA | RandomForest | 0.5342 | ⚠ Weak | Regime-sensitive, use with caution |
| TMGH_CA | LightGBM | 0.5264 | ⚠ Weak | Lowest signal, overfit gap still present |

### Recommended operating threshold per ticker

- COMI_CA → use **0.50**. The mathematically optimal threshold near 0.29 is too aggressive and degrades practical precision.
- HRHO_CA → use **0.50**. The fixed model is working well and the optimal threshold is effectively the default anyway.
- ORWE_CA → use **0.4743**. This is the only ticker where the optimised threshold is genuinely useful.
- SWDY_CA → use **0.50**. The optimal threshold near 0.36 is degenerate and too permissive.
- TMGH_CA → use **0.50**. The optimal threshold near 0.31 is degenerate and too permissive.

### What this means

- Threshold tuning improved F1, but only ORWE_CA produced a threshold that is clearly worth operationalising.
- HRHO_CA is the strongest corrected XGBoost ticker, but its best threshold is still effectively the default 0.50.
- COMI_CA, SWDY_CA, and TMGH_CA still have weak predictive edge, so the validation threshold should not be mistaken for true model quality.
- The final `.joblib` artifacts and `optimized_hyperparameters.json` now contain the selected production settings for each ticker.

In [2]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any, Dict, List, Tuple

import joblib
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    )
from xgboost import XGBClassifier

# ── Paths and shared configuration ───────────────────────────────────────────
def resolve_path(candidates: List[Path]) -> Path:
    for p in candidates:
        if p.exists():
            return p
    return candidates[0]


PROCESSED_DIR = resolve_path([Path("../data/processed"), Path("data/processed")])
MODELS_DIR = resolve_path([Path("../models"), Path("models")])
MODELS_DIR.mkdir(parents=True, exist_ok=True)
WINNER_CONFIG_PATH = MODELS_DIR / "winner_config.json"
OPTIMIZED_HP_PATH = MODELS_DIR / "optimized_hyperparameters.json"

FEATURE_COLUMNS: List[str] = [
    "RSI_14",
    "MACD_Norm", "MACD_Signal_Norm", "MACD_Hist_Norm",
    "Bollinger_Width", "Bollinger_Position",
    "ATR_Pct",
    "Return_Lag_1", "Return_Lag_5", "Return_Lag_10", "Return_Lag_21",
    "Close_MA5_Ratio", "Close_MA21_Ratio",
    "Close_CV5", "Close_CV21",
    "Volume_Ratio", "Volume_Spike",
    "Day_Of_Week", "Month", "is_Ramadan",
]
TARGET_COLUMN: str = "Target"
TRAIN_END: str = "2023-12-31"
VAL_END: str = "2024-12-31"
REGIME_SPLIT: str = "2024-06-30"
RANDOM_STATE: int = 42


def load_winner_config(path: Path) -> Dict[str, Dict[str, Any]]:
    if not path.exists():
        raise FileNotFoundError(f"Winner config not found at {path}.")
    with open(path, "r") as file_handle:
        return json.load(file_handle)


WINNER_CONFIG = load_winner_config(WINNER_CONFIG_PATH)
optimized_hp = json.loads(OPTIMIZED_HP_PATH.read_text())
TICKERS: List[str] = list(WINNER_CONFIG.keys())


def load_ticker_splits(
    ticker: str,
) -> Tuple[
    pd.DataFrame, pd.Series,
    pd.DataFrame, pd.Series,
    pd.DataFrame, pd.Series,
    pd.DataFrame, pd.Series,
    pd.DatetimeIndex,
]:
    path = PROCESSED_DIR / f"{ticker}_features.parquet"
    if not path.exists():
        raise FileNotFoundError(f"Parquet not found: {path}")

    df = pd.read_parquet(path).sort_index()
    train = df.loc[df.index <= TRAIN_END]
    val = df.loc[(df.index > TRAIN_END) & (df.index <= VAL_END)]

    def xy(subset: pd.DataFrame) -> Tuple[pd.DataFrame, pd.Series]:
        return (
            subset[FEATURE_COLUMNS].copy(),
            subset[TARGET_COLUMN].astype(int).copy(),
        )

    X_train, y_train = xy(train)
    X_val, y_val = xy(val)
    val_h1 = val.loc[val.index < REGIME_SPLIT]
    val_h2 = val.loc[val.index >= REGIME_SPLIT]
    X_val_h1, y_val_h1 = xy(val_h1)
    X_val_h2, y_val_h2 = xy(val_h2)

    return X_train, y_train, X_val, y_val, X_val_h1, y_val_h1, X_val_h2, y_val_h2, val.index


def rebuild_final_model(
    model_name: str,
    best_params: Dict[str, Any],
    scale_pos_weight: float | None = None,
) -> Any:
    shared_kwargs = dict(random_state=RANDOM_STATE, n_jobs=-1)

    if model_name == "XGBoost":
        xgb_params = dict(best_params)
        if scale_pos_weight is not None:
            xgb_params["scale_pos_weight"] = scale_pos_weight
        return XGBClassifier(
            **xgb_params,
            use_label_encoder=False,
            eval_metric="logloss",
            verbosity=0,
            **shared_kwargs,
        )
    if model_name == "LightGBM":
        return LGBMClassifier(
            **best_params,
            class_weight="balanced",
            verbose=-1,
            **shared_kwargs,
        )
    if model_name == "RandomForest":
        return RandomForestClassifier(
            **best_params,
            class_weight="balanced",
            **shared_kwargs,
        )
    raise ValueError(f"Unknown model_name: {model_name}")

## 2. Threshold Optimisation and scale_pos_weight Fix

This section retrains the selected models, applies the XGBoost class-balance fix where needed, and then finds the validation threshold that maximises F1 before persisting the final artifacts.

In [3]:
# ── Threshold helpers ────────────────────────────────────────────────────────

def find_optimal_threshold(
    model: Any,
    X_val: pd.DataFrame,
    y_val: pd.Series,
) -> Tuple[float, float]:
    """
    Find the probability threshold that maximises F1 on the validation set.
    Returns (best_threshold, best_f1).
    """
    y_proba = model.predict_proba(X_val)[:, 1]
    precisions, recalls, thresholds = precision_recall_curve(y_val, y_proba)
    f1_scores = (2 * precisions[:-1] * recalls[:-1]) / (
        precisions[:-1] + recalls[:-1] + 1e-9
    )
    best_idx = f1_scores.argmax()
    return float(thresholds[best_idx]), float(f1_scores[best_idx])


def compute_metrics_at_threshold(
    model: Any,
    X: pd.DataFrame,
    y: pd.Series,
    threshold: float,
) -> Dict[str, float]:
    """Evaluate a model using a custom probability threshold instead of 0.5."""
    y_proba = model.predict_proba(X)[:, 1]
    y_pred = (y_proba >= threshold).astype(int)
    return {
        "threshold": round(threshold, 4),
        "accuracy": round(accuracy_score(y, y_pred), 4),
        "precision": round(precision_score(y, y_pred, zero_division=0), 4),
        "recall": round(recall_score(y, y_pred, zero_division=0), 4),
        "f1": round(f1_score(y, y_pred, zero_division=0), 4),
        "roc_auc": round(roc_auc_score(y, y_proba), 4),
    }


def get_scale_pos_weight(y_train: pd.Series) -> float:
    """XGBoost's class_weight='balanced' equivalent."""
    n_down = int((y_train == 0).sum())
    n_up = int((y_train == 1).sum())
    return round(n_down / n_up, 4)

In [4]:
# ── Run threshold optimisation for all tickers ────────────────────────────────
threshold_results = []
optimal_thresholds: Dict[str, float] = {}

for ticker in TICKERS:
    model_name = WINNER_CONFIG[ticker]["model"]

    (
        X_train, y_train,
        X_val,   y_val,
        X_val_h1, y_val_h1,
        X_val_h2, y_val_h2,
        val_index,
    ) = load_ticker_splits(ticker)

    # ── For XGBoost tickers: retrain with scale_pos_weight fix ───────────────
    if model_name == "XGBoost":
        spw = get_scale_pos_weight(y_train)
        print(f"\n  {ticker} ({model_name}) — scale_pos_weight = {spw}")

        # Load best params from optimized_hyperparameters.json
        best_params = optimized_hp[ticker]["params"].copy()

        final_model = XGBClassifier(
            **best_params,
            scale_pos_weight=spw,  # ← THE FIX: was silently ignored before
            use_label_encoder=False,
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            verbosity=0,
            n_jobs=-1,
        )
        final_model.fit(X_train, y_train)
        print(f"  Retrained with scale_pos_weight={spw}")

    else:
        # LightGBM and RandomForest: reload the already-fitted model
        # (they handled class_weight correctly)
        best_params = optimized_hp[ticker]["params"].copy()
        final_model = rebuild_final_model(model_name, best_params)
        final_model.fit(X_train, y_train)

    # ── Find optimal threshold on validation set ──────────────────────────────
    best_threshold, best_f1_at_threshold = find_optimal_threshold(
        final_model, X_val, y_val
    )
    optimal_thresholds[ticker] = best_threshold

    # ── Compare default (0.5) vs optimal threshold ────────────────────────────
    metrics_050 = compute_metrics_at_threshold(final_model, X_val, y_val, 0.50)
    metrics_opt = compute_metrics_at_threshold(final_model, X_val, y_val, best_threshold)

    print(f"\n  {ticker} — {model_name}")
    print(f"  {'Metric':<14} {'Threshold=0.50':>16} {'Threshold={:.3f}'.format(best_threshold):>18}")
    print(f"  {'-'*50}")
    for metric in ["f1", "precision", "recall", "accuracy", "roc_auc"]:
        marker = " ←" if metric == "f1" else ""
        print(
            f"  {metric:<14} {metrics_050[metric]:>16.4f} "
            f"{metrics_opt[metric]:>18.4f}{marker}"
        )

    threshold_results.append({
        "ticker": ticker,
        "model": model_name,
        "default_f1": metrics_050["f1"],
        "default_precision": metrics_050["precision"],
        "default_recall": metrics_050["recall"],
        "optimal_threshold": best_threshold,
        "optimal_f1": metrics_opt["f1"],
        "optimal_precision": metrics_opt["precision"],
        "optimal_recall": metrics_opt["recall"],
        "roc_auc": metrics_opt["roc_auc"],  # same either way
    })

    # ── Save model artifact ───────────────────────────────────────────────────
    model_path = MODELS_DIR / f"{ticker}_best_model.joblib"
    joblib.dump(final_model, model_path)
    print(f"  Saved → {model_path}")


  COMI_CA (XGBoost) — scale_pos_weight = 1.0228
  Retrained with scale_pos_weight=1.0228

  COMI_CA — XGBoost
  Metric           Threshold=0.50    Threshold=0.295
  --------------------------------------------------
  f1                       0.5628             0.6591 ←
  precision                0.5652             0.4915
  recall                   0.5603             1.0000
  accuracy                 0.5738             0.4937
  roc_auc                  0.5234             0.5234
  Saved → models\COMI_CA_best_model.joblib

  HRHO_CA (XGBoost) — scale_pos_weight = 1.1799
  Retrained with scale_pos_weight=1.1799

  HRHO_CA — XGBoost
  Metric           Threshold=0.50    Threshold=0.496
  --------------------------------------------------
  f1                       0.6200             0.6797 ←
  precision                0.5225             0.5148
  recall                   0.7623             1.0000
  accuracy                 0.5190             0.5148
  roc_auc                  0.5502         

In [6]:
# ── Summary table and persistence ─────────────────────────────────────────────
threshold_df = pd.DataFrame(threshold_results)
print("\n\nThreshold Optimisation Summary")
display(
    threshold_df.style
    .background_gradient(subset=["optimal_f1", "roc_auc"], cmap="RdYlGn")
    .format({
        "default_f1": "{:.4f}",
        "optimal_threshold": "{:.4f}",
        "optimal_f1": "{:.4f}",
        "optimal_precision": "{:.4f}",
        "optimal_recall": "{:.4f}",
        "roc_auc": "{:.4f}",
    })
)


# ── Save thresholds alongside optimized hyperparameters ───────────────────────
for ticker in TICKERS:
    optimized_hp[ticker]["optimal_threshold"] = optimal_thresholds[ticker]

OPTIMIZED_HP_PATH.write_text(json.dumps(optimized_hp, indent=2))
print(f"\n✓ Updated optimized_hyperparameters.json with optimal thresholds.")
print(f"  Saved → {OPTIMIZED_HP_PATH.resolve()}")
print()
print(json.dumps({t: {"optimal_threshold": v} for t, v in optimal_thresholds.items()}, indent=2))



Threshold Optimisation Summary


,ticker,model,default_f1,default_precision,default_recall,optimal_threshold,optimal_f1,optimal_precision,optimal_recall,roc_auc
0,COMI_CA,XGBoost,0.5628,0.565200,0.560300,0.2949,0.6591,0.4915,1.0000,0.5234
1,HRHO_CA,XGBoost,0.6200,0.522500,0.762300,0.4957,0.6797,0.5148,1.0000,0.5502
2,ORWE_CA,LightGBM,0.5630,0.549200,0.577600,0.4743,0.6733,0.5455,0.8793,0.5895
3,SWDY_CA,RandomForest,0.6132,0.526900,0.733300,0.3570,0.6704,0.5064,0.9917,0.5342
4,TMGH_CA,LightGBM,0.4519,0.516500,0.401700,0.3108,0.6667,0.5114,0.9573,0.5264



✓ Updated optimized_hyperparameters.json with optimal thresholds.
  Saved → D:\my_projects\Egx-analyst\notebooks\models\optimized_hyperparameters.json

{
  "COMI_CA": {
    "optimal_threshold": 0.2948729693889618
  },
  "HRHO_CA": {
    "optimal_threshold": 0.495667427778244
  },
  "ORWE_CA": {
    "optimal_threshold": 0.47434952877280934
  },
  "SWDY_CA": {
    "optimal_threshold": 0.3570048486253765
  },
  "TMGH_CA": {
    "optimal_threshold": 0.31082823367922136
  }
}
